### Qdrant

In [ ]:
#sudo docker run -p 6333:6333 -p 6334:6334 -v $(pwd)/qdrant_storage:/qdrant/storage:z qdrant/qdrant

In [9]:
import os
import json
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import uuid
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
cwd = os.getcwd()

just doing one single policy (it's like 1000 pages tho)

In [2]:
with open(f"{cwd}/data/ie_cap.jsonl", "r", encoding="utf-8") as f:
    json_list = list(f)

cap_set= []
for json_str in json_list:
    result = json.loads(json_str)
    cap_set.append(result)

In [18]:
client = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "irish_cap"
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )

/home/marwas/keymetrics/kmavenv/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.14.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [13]:
# first we need to split the long chunks into smaller chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

def chunk_chunks(chunks):
    chunked_chunks = []
    for c in chunks:
        # if our text is longer than we want, chunk it
        if len(c['text']) > 1500:
            sub_chunks = splitter.split_text(c['text'])
            for i, sub_txt in enumerate(sub_chunks):
                # new mini chunk version
                new_chunk = c.copy() #copies all data first, then we can update text and id
                new_chunk['text'] = sub_txt
                new_chunk['chunk_id'] = f"{c['chunk_id']}_sub_{i}"
                chunked_chunks.append(new_chunk)
        else:
            chunked_chunks.append(c)
    return chunked_chunks

In [14]:
chunked_cap_set = chunk_chunks(cap_set)
print(len(cap_set), len(chunked_cap_set))

4446 4554


In [4]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device="cuda:2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8421.79it/s]


In [15]:
def upload_existing_chunks(chunks, batch_size=100):
    print(f"{len(chunks)} to process")
    texts = [c['text'] for c in chunks]
    embeddings = model.encode(texts, show_progress_bar=True)
    all_points = []
    for i, chunk in enumerate(chunks):
        point_id = str(uuid.uuid4())
        all_points.append(PointStruct(
            id=point_id,
            vector=embeddings[i].tolist(),
            payload={
                "text": chunk['text'],
                "page": chunk['pg'],
                "original_chunk_id": chunk['chunk_id'],
                "metadata": chunk.get('label', [])
            }
        ))
    print(f"uploading (in batches so we dont timeout)")
    for i in tqdm(range(0, len(all_points), batch_size)):
        batch = all_points[i : i + batch_size]
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=batch,
            wait=True
        )
    print(f"Done")

In [16]:
upload_existing_chunks(chunked_cap_set)

4554 to process


Batches: 100%|██████████| 143/143 [00:19<00:00,  7.39it/s]


uploading (in batches so we dont timeout)


100%|██████████| 46/46 [00:03<00:00, 14.80it/s]

Done
